---
title: "Motorcycle Parts Sales Analysis"
author: "Your Name"
format:
  html:
    theme: flatly
    toc: true
    code-fold: true
---

## Executive Summary
The board of directors requested an analysis of our wholesale revenue streams. The company operates three regional warehouses and offers various payment methods, each incurring a different percentage fee. 

**Objective:** Calculate the net revenue for each product line, grouped by month and warehouse, exclusively for our **Wholesale** clients.

In [ ]:
import duckdb
import pandas as pd

# Connect to a temporary local database
con = duckdb.connect(database=':memory:')

# Let's peek at the first 3 rows of our data to understand the schema
preview_query = "SELECT * FROM 'sales.csv' LIMIT 3"
con.execute(preview_query).df()

In [ ]:
sql_query = """
SELECT 
    product_line,
    CASE 
        WHEN EXTRACT(MONTH FROM CAST(date AS DATE)) = 6 THEN 'June'
        WHEN EXTRACT(MONTH FROM CAST(date AS DATE)) = 7 THEN 'July'
        WHEN EXTRACT(MONTH FROM CAST(date AS DATE)) = 8 THEN 'August'
    END AS month,
    warehouse,
    ROUND(SUM(total * (1 - payment_fee)), 2) AS net_revenue
FROM 'sales.csv'
WHERE client_type = 'Wholesale'
GROUP BY product_line, month, warehouse
ORDER BY product_line, month, net_revenue DESC;
"""

# Execute the query and save the result as a styled dataframe
net_revenue_df = con.execute(sql_query).df()

# Display the final, clean table
net_revenue_df

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Set aesthetic styling
sns.set_theme(style="whitegrid")
plt.figure(figsize=(10, 6))

# Create a bar chart showing revenue by product line, split by warehouse
sns.barplot(
    data=net_revenue_df, 
    x="net_revenue", 
    y="product_line", 
    hue="warehouse",
    palette="viridis"
)

plt.title("Wholesale Net Revenue by Product Line & Warehouse", fontsize=14, weight='bold')
plt.xlabel("Net Revenue ($)", fontsize=12)
plt.ylabel("Product Line", fontsize=12)
plt.show()